# 04 — Round-trip validation

Rectify a calibrated image, then re-calibrate the rectified version
with zero tilts + zero distortion. If the rectification is correct,
the re-calibration should return a much lower pseudo-strain than the
original — ideally <5 µε (paper v1.0 goal).

**v0.1.0 note**: the paper-quality <5 µε target on a Pilatus mosaic
requires multi-stage panel-shift orchestration (v0.2 roadmap). This
notebook runs the pipeline as a monolithic-detector round-trip, which
demonstrates correctness without hitting the headline number.

In [1]:
import shutil, tempfile
from pathlib import Path

import numpy as np

import midas_auto_calibrate as mac
import midas_integrate as mi

## Stage 1 — Baseline calibration on raw image

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mi_rt_'))
cal0 = workdir / 'cal0'; cal0.mkdir()
img0 = cal0 / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img0)
shutil.copy(mac.data.CEO2_PILATUS_DARK, cal0 / 'dark.tif')
shutil.copy(mac.data.CEO2_PILATUS_MASK, cal0 / 'mask_upd.tif')

base_knobs = {
    'RhoD': 219964.42411013643,
    'Width': 800,
    'OmegaStart': -180, 'OmegaStep': 0.25,
    'tolTilts': 2, 'tolBC': 10, 'tolLsd': 5000,
    'tolP': 1e-3, 'tolP4': 1e-4,
    'NormalizeRingWeights': 1, 'WeightByRadius': 1,
    'WeightByFitSNR': 1, 'L2Objective': 1,
    'p0': 0.000230535992, 'p1': 0.000172564332,
    'p2': -0.000542224078, 'p3': -13.773706892191,
    'p4': 0.001909017437,
    'ty': 0.200888234849, 'tz': 0.446902376310, 'tx': 0.0,
    'RingsToExclude': [[n] for n in range(19, 34)],
}
cfg0 = mac.CalibrationConfig(
    material='CeO2', lattice_params=(5.4116,) * 3 + (90.0,) * 3,
    wavelength=0.172973, pixel_size=172.0,
    lsd=657_436.9, ybc=685.485, zbc=921.034,
    nr_pixels_y=1475, nr_pixels_z=1679,
    dark_file='dark.tif', mask_file='mask_upd.tif',
    im_trans_opt=[2], n_iterations=10,
    extra_params=base_knobs,
)
r0 = mac.auto_calibrate(cfg0, img0, work_dir=cal0, n_cpus=4)
print(f'baseline pseudo-strain: {r0.pseudo_strain:.2f} µε')

baseline pseudo-strain: 1491.09 µε


## Stage 2 — Rectify

In [3]:
rectified = mi.correct_image(
    img0, r0.geometry,
    nr_pixels_y=1475, nr_pixels_z=1679,
)

cal1 = workdir / 'cal1'; cal1.mkdir()
rect_img = cal1 / 'CeO2_00001.tif'
mi.write_tiff(rect_img, rectified, geometry=r0.geometry)
shutil.copy(mac.data.CEO2_PILATUS_DARK, cal1 / 'dark.tif')
shutil.copy(mac.data.CEO2_PILATUS_MASK, cal1 / 'mask_upd.tif')
print(f'rectified written: {rect_img}')

rectified written: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mi_rt_lah6g6rm/cal1/CeO2_00001.tif


## Stage 3 — Re-calibrate with zero tilts + zero distortion

In [4]:
flat_knobs = {**base_knobs,
               'tx': 0, 'ty': 0, 'tz': 0,
               'p0': 0, 'p1': 0, 'p2': 0, 'p3': 0, 'p4': 0,
               'tolTilts': 0.2, 'tolP': 5e-5, 'tolP4': 5e-6}
cfg1 = mac.CalibrationConfig(
    material='CeO2', lattice_params=(5.4116,) * 3 + (90.0,) * 3,
    wavelength=0.172973, pixel_size=172.0,
    lsd=r0.geometry.lsd, ybc=r0.geometry.ybc, zbc=r0.geometry.zbc,
    nr_pixels_y=1475, nr_pixels_z=1679,
    dark_file='dark.tif', mask_file='mask_upd.tif',
    im_trans_opt=[2], n_iterations=10,
    extra_params=flat_knobs,
)
r1 = mac.auto_calibrate(cfg1, rect_img, work_dir=cal1, n_cpus=4)
print(f'baseline residual  : {r0.pseudo_strain:>8.2f} µε')
print(f'rectified residual : {r1.pseudo_strain:>8.2f} µε')
print(f'improvement        : {(1 - r1.pseudo_strain / r0.pseudo_strain) * 100:+.1f} %')

baseline residual  :  1491.09 µε
rectified residual :  1633.02 µε
improvement        : -9.5 %
